# Unified Road Defect Dataset

RoadGuard AI · Dataset validation and exploratory analysis

## 1. Dataset overview

This notebook reads saved Stage 2 reports. The publisher-pair counts support data auditing. Analysis charts and visual examples use the prepared training partition.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

from roadguard.config import load_config
from roadguard.data.unified_workflow import load_unified_report
from roadguard.utils.paths import project_root

root = project_root()
config = load_config(root, profile=root / "configs/data/unified.yaml")
report = load_unified_report(root, config)
prepared = bool(report and report["status"] == "prepared")
eda = report.get("training_eda") if report else None

if report:
    display(
        pd.Series(
            {
                "Repository": report["source_repository"],
                "Revision": report["revision"],
                "Status": report["status"],
                "Discovered images": report["images_found"],
                "Discovered labels": report["labels_found"],
                "Valid image-label pairs": report["validated_pairs"],
                "Prepared fingerprint": report["fingerprint"],
            },
            name="Dataset",
        )
    )
    if report["validation_issue_counts"]:
        display(pd.Series(report["validation_issue_counts"], name="Publisher findings"))
    if report.get("duplicate_resolution"):
        print("Reviewed duplicate exclusion", root / report["duplicate_resolution"])
    if report.get("prepared_validation"):
        check = report["prepared_validation"]
        print("Prepared validation", "passed" if check["passed"] else "blocked")
else:
    print("Run the Stage 2 acquisition and validation commands in README.md to generate a report.")

## 2. Split statistics

The project uses a seed-42 split with approximately 80% training, 10% validation and 10% held-out test. Sample review and EDA use training images. Keep the test partition sealed for final evaluation.

In [ ]:
if prepared:
    display(
        pd.DataFrame(
            {
                name: {key: stats[key] for key in ("images", "objects", "negatives")}
                for name, stats in report["splits"].items()
            }
        ).T
    )
elif report:
    print("Project splits are pending successful dataset validation.")

## 3. Class distribution

In [ ]:
if eda:
    stats = eda["statistics"]
    display(
        pd.DataFrame(
            {
                "Annotations": stats["instances_per_class"],
                "Images": stats["images_per_class"],
                "% of annotations": {k: 100 * v for k, v in stats["instance_share"].items()},
                "% of images": {k: 100 * v for k, v in stats["image_share"].items()},
            }
        ).round(2)
    )
elif report:
    display(
        pd.Series(
            report["input_pair_statistics"]["instances_per_class"],
            name="Annotations in accepted publisher pairs",
        )
    )

## 4. Sources and countries

In [ ]:
if report:
    audit = eda["statistics"] if eda else report["input_pair_statistics"]
    for key in ("by_source", "by_country", "by_view"):
        display(Markdown(f"**{key.replace('_', ' ').title()}**"))
        display(
            pd.DataFrame(
                {
                    name: {
                        "Images": row["images"],
                        "Objects": row["objects"],
                        **row["instances_per_class"],
                    }
                    for name, row in audit[key].items()
                }
            ).T
        )

## 5. Image resolution

In [ ]:
def show_figure(name):
    if eda:
        for relative_path in eda["figures"]:
            file = root / relative_path
            if file.name == name and file.is_file():
                display(Image(filename=str(file)))
                break


if eda:
    display(
        pd.DataFrame(
            {
                key: eda["statistics"][key]
                for key in ("image_width_px", "image_height_px", "image_aspect_ratio")
            }
        ).T
    )
    show_figure("geometry.png")

## 6. Objects per image

In [ ]:
if eda:
    stats = eda["statistics"]
    display(
        pd.Series(
            {
                "Empty images": stats["negatives"],
                "Multi-object images": stats["multi_object_images"],
                "Multi-class images": stats["multi_class_images"],
            },
            name="Training partition",
        )
    )
    show_figure("composition.png")

## 7. Bounding-box quality

Thresholds identify candidates for visual review. Original labels remain intact.

In [ ]:
if report:
    quality = eda["box_diagnostics"] if eda else report.get("box_diagnostics")
    if quality:
        geometry = eda["statistics"] if eda else report["input_pair_statistics"]
        display(pd.DataFrame(geometry["box_geometry"]).T)
        display(pd.Series(quality["thresholds"], name="Diagnostic thresholds"))
        display(pd.Series(quality["flags"], name="Box flag counts", dtype="int64"))
        print(root / (eda["review_csv"] if eda else report["review_csv"]))

## 8. Visual review

Use the per-class and per-source sheets for representative cases. The ranked sheets illustrate dense scenes and extreme box geometry.

In [ ]:
if eda:
    for relative_path in eda["figures"]:
        file = root / relative_path
        if file.name not in {"geometry.png", "composition.png"} and file.is_file():
            display(Image(filename=str(file)))

## 9. Class imbalance

In [ ]:
if eda:
    stats = eda["statistics"]
    ratios = {"training": stats["largest_to_smallest_class_ratio"]}
    ratios.update(
        {
            source: row["largest_to_smallest_class_ratio"]
            for source, row in stats["by_source"].items()
        }
    )
    display(pd.Series(ratios, name="Largest / smallest class count").round(2))

## 10. Risks and training recommendations

Use training and validation evidence for future model decisions.

In [ ]:
if report:
    for section in ("key_risks", "training_recommendations"):
        display(Markdown("\n".join(f"- {item}" for item in report[section])))